# باب 7: چیٹ ایپلیکیشنز بنانا
## مائیکروسافٹ فاؤنڈری ماڈلز API کوئیک اسٹارٹ

یہ نوٹ بک [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) سے مطابقت حاصل کی گئی ہے جس میں نوٹ بکس شامل ہیں جو [Azure OpenAI](notebook-azure-openai.ipynb) خدمات تک رسائی دیتی ہیں۔

> **نوٹ:** GitHub ماڈلز جولائی 2026 کے آخر میں ریٹائر ہو رہے ہیں۔ یہ نوٹ بک اب [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst) کو ہدف بناتی ہے، جو وہی مفت آزمائش ماڈل کیٹلاگ اور Azure AI انفرنس SDK تجربہ پیش کرتا ہے۔


# جائزہ  
"بڑے زبان ماڈلز ایسے فنکشن ہیں جو متن کو متن میں نقش کرتے ہیں۔ دیے گئے متنی سلسلے کے لیے، ایک بڑا زبان ماڈل کوشش کرتا ہے کہ اگلا متن کیا ہوگا اسے پیش گوئی کرے"(1). یہ "کوئیک اسٹارٹ" نوٹ بک صارفین کو اعلیٰ سطح کے LLM تصورات، AML کے ساتھ شروع کرنے کے لیے بنیادی پیکیج کی ضروریات، پرامپٹ ڈیزائن کا نرم تعارف، اور مختلف استعمال کے کیسز کی چند مختصر مثالوں سے متعارف کرائے گا۔ 


## مواد کی فہرست  

[جائزہ](#overview)  
[OpenAI سروس استعمال کرنے کا طریقہ](#how-to-use-openai-service)  
[1. اپنی OpenAI سروس بنانا](#1.-creating-your-openai-service)  
[2. تنصیب](#2.-installation)    
[3. اسناد](#3.-credentials)  

[استعمال کے کیسز](#use-cases)    
[1. متن کا خلاصہ بنائیں](#1.-summarize-text)  
[2. متن کو درجہ بندی کریں](#2.-classify-text)  
[3. نئے مصنوعات کے نام پیدا کریں](#3.-generate-new-product-names)  
[4. کلاسفائر کی باریکی سے تربیت کریں](#4.fine-tune-a-classifier)  

[حوالہ جات](#references)


### اپنا پہلا پرامپٹ بنائیں  
یہ مختصر مشق Microsoft Foundry Models میں ایک ماڈل کو ایک سادہ کام "خلاصہ سازی" کے لیے پرامپٹ جمع کرانے کا بنیادی تعارف فراہم کرے گی۔  


**اقدامات**:  
1. اگر آپ نے ابھی تک نہیں کیا تو اپنے پائیتھن ماحول میں `azure-ai-inference` لائبریری انسٹال کریں۔  
2. معیاری معاون لائبریریاں لوڈ کریں اور اپنے Microsoft Foundry Models کے اسناد مرتب کریں۔  
3. اپنے کام کے لیے ایک ماڈل منتخب کریں  
4. ماڈل کے لیے ایک سادہ پرامپٹ بنائیں  
5. اپنی درخواست ماڈل API کو جمع کرائیں!  


### 1. `azure-ai-inference` انسٹال کریں


In [ ]:
%pip install azure-ai-inference

### 2. معاون لائبریریاں درآمد کریں اور اسناد قائم کریں


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. درست ماڈل تلاش کرنا  
GPT-4o اور GPT-4o mini جیسے ماڈل قدرتی زبان کو سمجھ اور پیدا کر سکتے ہیں، اور یہ Microsoft Foundry Models کے کیٹلاگ میں Meta، Mistral، Cohere، اور Microsoft کے ماڈلز کے ساتھ دستیاب ہیں۔


In [ ]:
# Select a general purpose chat model
model_name = "gpt-5-mini"


## 4. پرامپٹ ڈیزائن  

"بڑے زبان کے ماڈلز کا جادو یہ ہے کہ بے شمار متون پر اس پیش گوئی کی غلطی کو کم سے کم کرنے کی تربیت حاصل کرکے، یہ ماڈلز ان پیش گوئیوں کے لیے مفید تصورات سیکھ لیتے ہیں۔ مثال کے طور پر، وہ ایسے تصورات سیکھتے ہیں جیسے"(1):

* ہجے کیسے کریں
* گرامر کیسے کام کرتی ہے
* پیرایہ بیان کیسے کریں
* سوالات کے جواب کیسے دیں
* بات چیت کیسے کریں
* کئی زبانوں میں لکھنا کیسے ہے
* کوڈنگ کیسے کریں
* وغیرہ۔

#### ایک بڑے زبان کے ماڈل کو کیسے کنٹرول کیا جائے  
"بڑے زبان کے ماڈل میں تمام ان پٹس میں سب سے زیادہ اثر انداز ہونے والا ٹیکسٹ پرامپٹ ہے(1).

بڑے زبان کے ماڈلز کو چند طریقوں سے آؤٹ پٹ پیدا کرنے کے لئے پرامپٹ کیا جا سکتا ہے:

ہدایت: ماڈل کو بتائیں کہ آپ کیا چاہتے ہیں
تکمیل: ماڈل کو اس کے شروع کو مکمل کرنے پر مجبور کریں جو آپ چاہتے ہیں
مظاہرہ: ماڈل کو دکھائیں کہ آپ کیا چاہتے ہیں، یا تو:
پرامپٹ میں چند مثالیں
ایک فائن ٹیوننگ تربیتی ڈیٹاسیٹ میں سینکڑوں یا ہزاروں مثالیں"



#### پرامپٹ بنانے کے تین بنیادی اصول ہیں:

**دکھائیں اور بتائیں**۔ واضح کریں کہ آپ کیا چاہتے ہیں چاہے وہ ہدایات کے ذریعے ہو، مثالوں کے ذریعے، یا دونوں کے امتزاج سے۔ اگر آپ چاہتے ہیں کہ ماڈل اشیاء کی فہرست کو حروف تہجی کے ترتیب میں رکھے یا کسی پیراگراف کو جذبات کی بنیاد پر درجہ بند کرے، تو اسے دکھائیں کہ یہی آپ کی خواہش ہے۔

**معیاری ڈیٹا فراہم کریں**۔ اگر آپ ایک درجہ بندی کار بنانے کی کوشش کر رہے ہیں یا ماڈل کو کسی پیٹرن کی پیروی کرانا چاہتے ہیں، تو یقینی بنائیں کہ کافی مثالیں موجود ہوں۔ اپنی مثالوں کو غور سے پڑھیں—ماڈل عام طور پر بنیادی ہجوں کی غلطیوں کو سمجھ سکتا ہے اور جواب دے سکتا ہے، لیکن یہ بھی فرض کر سکتا ہے کہ یہ جان بوجھ کر کیا گیا ہے اور یہ جواب کو متاثر کر سکتا ہے۔

**اپنی ترتیبات چیک کریں۔** درجہ حرارت اور top_p کی ترتیبات اس بات کو کنٹرول کرتی ہیں کہ ماڈل جواب پیدا کرنے میں کتنا تعیناتی ہوتا ہے۔ اگر آپ ایسے جواب کے لئے پوچھ رہے ہیں جس کا صرف ایک درست جواب ہو، تو آپ کو یہ ترتیبات کم رکھنی چاہئیں۔ اگر آپ مزید مختلف جوابات چاہتے ہیں، تو آپ انہیں زیادہ رکھنا چاہیں گے۔ ان ترتیبات کے ساتھ لوگوں کی سب سے بڑی غلطی یہ سمجھنا ہے کہ یہ "ذہانت" یا "تخلیقی صلاحیت" کے کنٹرول ہیں۔


ماخذ: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. جمع کروائیں!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### ایک ہی کال کو دہرائیں، نتائج کیسے موازنہ کرتے ہیں؟


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## متن کا خلاصہ کریں  
#### چیلنج  
ایک ٹیکسٹ پیراگراف کے آخر میں 'tl;dr:' شامل کرکے متن کا خلاصہ کریں۔ غور کریں کہ ماڈل بغیر کسی اضافی ہدایات کے مختلف کاموں کو کیسے سمجھتا ہے۔ آپ ماڈل کے رویے کو تبدیل کرنے اور حاصل کردہ خلاصے کو حسب ضرورت بنانے کے لیے tl;dr سے زیادہ وضاحتی پرامپٹس کے ساتھ تجربہ کر سکتے ہیں(3)۔  

حالیہ کاموں نے بڑے ٹیکسٹ کورپس پر پری-ٹریننگ اور پھر مخصوص کام پر فن ٹیوننگ کے ذریعے کئی NLP کاموں اور بینچ مارکس پر نمایاں ترقی دکھائی ہے۔ اگرچہ عام طور پر اس کا ڈھانچہ ٹاسک-ایگناسٹک ہوتا ہے، لیکن اس طریقہ کار کے لیے ہزاروں یا دس ہزاروں مثالوں کے ٹاسک-مخصوص فن ٹیوننگ ڈیٹاسیٹس کی ضرورت ہوتی ہے۔ اس کے برعکس، انسان عام طور پر چند مثالوں یا آسان ہدایات سے نئی زبان کے کام انجام دے سکتے ہیں - کچھ ایسا جس میں موجودہ NLP نظام ابھی بھی زیادہ تر جدوجہد کرتے ہیں۔ یہاں ہم دکھاتے ہیں کہ زبان کے ماڈلز کو بڑھانے سے کام-ایگناسٹک، چند شوٹ کارکردگی میں بہتری آتی ہے، جو بعض اوقات پچھلے جدید فن ٹیوننگ طریقوں کے ساتھ مقابلہ کر سکتی ہے۔  



خلاصہ  


# مختلف استعمال کے معاملات کے لیے مشقیں  
1. متن کا خلاصہ بنائیں  
2. متن کی درجہ بندی کریں  
3. نئے مصنوعات کے نام بنائیں


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## متن کو درجہ بندی کریں  
#### چیلنج  
اشیاء کو ان زمروں میں درجہ بندی کریں جو انفرنس کے وقت مہیا کیے جائیں۔ درج ذیل مثال میں، ہم دونوں زمروں اور درجہ بندی کے لیے متن کو پرامپٹ (*playground_reference) میں فراہم کرتے ہیں۔  

صارف کی استفسار: ہیلو، میری لیپ ٹاپ کی بورڈ کی ایک چابی حال ہی میں ٹوٹ گئی ہے اور مجھے اس کا متبادل درکار ہے:  

درجہ بند شدہ زمرہ:  


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## نئے مصنوعات کے نام بنائیں
#### چیلنج
مثال کے الفاظ سے مصنوعات کے نام بنائیں۔ یہاں ہم پراڈکٹ کے بارے میں معلومات پرامپٹ میں شامل کرتے ہیں جس کے لیے ہم نام بنانا چاہتے ہیں۔ ہم ایک مشابہ مثال بھی فراہم کرتے ہیں تاکہ وہ پیٹرن دکھائیں جو ہم حاصل کرنا چاہتے ہیں۔ ہم نے درجہ حرارت کی قدر بھی زیادہ مقرر کی ہے تاکہ بے ترتیبیت اور زیادہ جدید جوابات حاصل ہوں۔

مصنوعات کی تفصیل: ایک گھریلو ملک شیک بنانے والا
بیج الفاظ: تیز، صحت مند، کمپیکٹ۔
مصنوعات کے نام: HomeShaker, Fit Shaker, QuickShake, Shake Maker

مصنوعات کی تفصیل: جوتوں کی ایک جوڑی جو کسی بھی جوتے کے سائز میں فٹ ہو سکتی ہے۔
بیج الفاظ: ہم آہنگ، فٹ، اومنی فٹ۔


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# حوالہ جات  
- [اوپن اے آئی کوک بک](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [مائیکروسافٹ فاؤنڈری پورٹل](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [متن کی درجہ بندی کے لیے GPT ماڈلز کو بہتر بنانے کے لیے بہترین طریقے](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# مزید مدد کے لیے  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# شرکاء
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
